# t-SNE Dimensionality Reduction - MNIST Digit Visualization
**Module 2 - Dimensionality Reduction - Assignment 2**

---

### Objective
Apply t-SNE to reduce 784-dimensional MNIST digit images to 2D for visualization, compare with PCA, and study the effect of hyperparameters (perplexity, iterations, learning rate).

### Dataset
**MNIST in CSV** (Kaggle)  
Download: https://www.kaggle.com/datasets/oddrationale/mnist-in-csv  
Place as: `data/mnist_train.csv`

### Topics Covered
- MNIST pixel data exploration and sample visualization
- PCA pre-reduction before t-SNE (standard best practice)
- t-SNE algorithm intuition (high-dim to low-dim probability matching)
- KL divergence as the optimization objective
- Perplexity hyperparameter comparison (5, 30, 50, 100)
- Iteration and learning rate effects
- PCA vs t-SNE side-by-side comparison
- KNN downstream classification on embeddings
- Model serialization

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

BG     = '#0f0a1e'
AX_BG  = '#1a0f2e'
ACCENT = '#c084fc'
GOLD   = '#fde68a'

DIGIT_COLORS = [
    '#f87171','#fb923c','#fbbf24','#a3e635',
    '#34d399','#22d3ee','#60a5fa','#a78bfa',
    '#f472b6','#94a3b8'
]

plt.rcParams['figure.facecolor'] = BG
plt.rcParams['axes.facecolor']   = AX_BG
plt.rcParams['axes.edgecolor']   = '#2d1f4e'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = 'white'
plt.rcParams['xtick.color']      = 'white'
plt.rcParams['ytick.color']      = 'white'
plt.rcParams['grid.color']       = '#1a0f2e'

print('Libraries imported.')

## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('../data/mnist_train.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
label_col = df.columns[0]
y_all = df[label_col].values
X_all = df.drop(columns=[label_col]).values.astype(np.float32)

print(f'Pixel features : {X_all.shape[1]}  (28x28 = {28*28})')
print(f'Total samples  : {X_all.shape[0]:,}')
print(f'Classes (digits): {np.unique(y_all)}')
print(f'Class distribution:')
for d in range(10):
    print(f'  Digit {d}: {(y_all==d).sum():5d} samples')

In [ ]:
# Visualize sample digits
fig, axes = plt.subplots(2, 10, figsize=(16, 4))
fig.suptitle('Sample MNIST Digits (2 per class)', color='white', fontsize=13, fontweight='bold')

for digit in range(10):
    digit_idx = np.where(y_all == digit)[0]
    for row in range(2):
        img = X_all[digit_idx[row]].reshape(28, 28)
        ax  = axes[row, digit]
        ax.imshow(img, cmap='inferno', interpolation='nearest')
        ax.axis('off')
        if row == 0:
            ax.set_title(str(digit), color=DIGIT_COLORS[digit],
                         fontsize=12, fontweight='bold')

plt.tight_layout()
plt.savefig('../static/digit_grid.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved digit_grid.png')

## 3. Sampling and Preprocessing

In [ ]:
# t-SNE is O(n^2) in memory - sample for tractability
N_SAMPLE = 5000
np.random.seed(42)
idx      = np.random.choice(len(X_all), size=N_SAMPLE, replace=False)
X_sample = X_all[idx]
y_sample = y_all[idx]
print(f'Working with {N_SAMPLE} samples')

# Scale
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_sample)
print('Scaled shape:', X_scaled.shape)

## 4. PCA Pre-reduction (Best Practice Before t-SNE)

In [ ]:
# Why: reduces noise and speeds up t-SNE significantly
# Standard: reduce to 50 dims before running t-SNE

PCA_DIMS = 50
pca_pre  = PCA(n_components=PCA_DIMS, random_state=42)
X_pca50  = pca_pre.fit_transform(X_scaled)
pca_var  = pca_pre.explained_variance_ratio_.sum()

print(f'PCA pre-reduction: {X_scaled.shape[1]} -> {PCA_DIMS} dims')
print(f'Variance retained: {pca_var*100:.1f}%')
print(f'Speedup benefit  : t-SNE on 50 dims is much faster than on 784')

In [ ]:
# 2D PCA for comparison with t-SNE later
pca_2d  = PCA(n_components=2, random_state=42)
X_pca2d = pca_2d.fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(10, 7))
for digit in range(10):
    mask = y_sample == digit
    ax.scatter(X_pca2d[mask, 0], X_pca2d[mask, 1],
               c=DIGIT_COLORS[digit], label=str(digit),
               alpha=0.6, s=12, edgecolors='none')
ax.set_title('PCA 2D - MNIST (digits overlap heavily)',
             color=ACCENT, fontsize=13, fontweight='bold')
ax.set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
ax.legend(title='Digit', facecolor=AX_BG, labelcolor='white',
          fontsize=9, markerscale=2, loc='upper right')
ax.grid(True, linestyle='--', alpha=0.15)
plt.tight_layout()
plt.show()

## 5. t-SNE - Default Settings (Perplexity=30)

In [ ]:
print('Running t-SNE... (may take 1-3 minutes on 5000 samples)')

tsne = TSNE(
    n_components  = 2,
    perplexity    = 30,       # effective number of neighbours
    n_iter        = 1000,     # optimisation iterations
    learning_rate = 'auto',   # auto = n_samples/early_exaggeration
    init          = 'pca',    # pca init is more stable than random
    random_state  = 42,
    n_jobs        = -1
)
X_tsne = tsne.fit_transform(X_pca50)

print(f'Output shape    : {X_tsne.shape}')
print(f'KL divergence   : {tsne.kl_divergence_:.4f}  (lower = better fit)')
print(f'n_iter_         : {tsne.n_iter_}')

In [ ]:
fig, ax = plt.subplots(figsize=(11, 9))

for digit in range(10):
    mask = y_sample == digit
    ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1],
               c=DIGIT_COLORS[digit], label=str(digit),
               alpha=0.7, s=18, edgecolors='none')

ax.set_title('t-SNE 2D Projection of MNIST Digits',
             color=ACCENT, fontsize=14, fontweight='bold')
ax.set_xlabel('t-SNE Dimension 1')
ax.set_ylabel('t-SNE Dimension 2')
ax.legend(title='Digit', facecolor=AX_BG, labelcolor='white',
          fontsize=10, markerscale=2, loc='upper right')
ax.grid(True, linestyle='--', alpha=0.15)
plt.tight_layout()
plt.savefig('../static/tsne_plot.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved tsne_plot.png')

## 6. PCA vs t-SNE Side-by-Side

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

for digit in range(10):
    mask = y_sample == digit
    axes[0].scatter(X_pca2d[mask, 0], X_pca2d[mask, 1],
                    c=DIGIT_COLORS[digit], label=str(digit),
                    alpha=0.6, s=12, edgecolors='none')
axes[0].set_title('PCA 2D Projection', color=ACCENT, fontsize=13, fontweight='bold')
axes[0].set_xlabel(f'PC1 ({pca_2d.explained_variance_ratio_[0]*100:.1f}%)')
axes[0].set_ylabel(f'PC2 ({pca_2d.explained_variance_ratio_[1]*100:.1f}%)')
axes[0].legend(title='Digit', facecolor=AX_BG, labelcolor='white',
               fontsize=9, markerscale=2, loc='upper right')
axes[0].grid(True, linestyle='--', alpha=0.15)

for digit in range(10):
    mask = y_sample == digit
    axes[1].scatter(X_tsne[mask, 0], X_tsne[mask, 1],
                    c=DIGIT_COLORS[digit], label=str(digit),
                    alpha=0.7, s=12, edgecolors='none')
axes[1].set_title('t-SNE 2D Projection', color=ACCENT, fontsize=13, fontweight='bold')
axes[1].set_xlabel('t-SNE Dim 1'); axes[1].set_ylabel('t-SNE Dim 2')
axes[1].legend(title='Digit', facecolor=AX_BG, labelcolor='white',
               fontsize=9, markerscale=2, loc='upper right')
axes[1].grid(True, linestyle='--', alpha=0.15)

plt.suptitle('PCA vs t-SNE: MNIST Digit Separation',
             color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/pca_vs_tsne.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved pca_vs_tsne.png')

## 7. Perplexity Hyperparameter Study

In [ ]:
# Use subsample for speed
N_PERP  = 2000
idx2    = np.random.choice(N_SAMPLE, size=N_PERP, replace=False)
X_perp  = X_pca50[idx2]
y_perp  = y_sample[idx2]

perplexities = [5, 30, 50, 100]
tsne_results = {}
kl_divs      = {}

print('Running perplexity comparison (4 t-SNE runs)...')
for p in perplexities:
    print(f'  perplexity={p}...')
    t = TSNE(n_components=2, perplexity=p, n_iter=500,
             learning_rate='auto', init='pca', random_state=42)
    emb = t.fit_transform(X_perp)
    tsne_results[p] = emb
    kl_divs[p]      = round(float(t.kl_divergence_), 4)
    print(f'    KL divergence: {kl_divs[p]:.4f}')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes = axes.flatten()

for ax, p in zip(axes, perplexities):
    emb = tsne_results[p]
    for digit in range(10):
        mask = y_perp == digit
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=DIGIT_COLORS[digit], label=str(digit),
                   alpha=0.7, s=14, edgecolors='none')
    ax.set_title(f'Perplexity = {p}  |  KL = {kl_divs[p]:.4f}',
                 color=ACCENT, fontsize=12, fontweight='bold')
    ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
    ax.legend(title='Digit', facecolor=AX_BG, labelcolor='white',
              fontsize=8, markerscale=1.5, loc='upper right')
    ax.grid(True, linestyle='--', alpha=0.15)

plt.suptitle('t-SNE Perplexity Comparison (MNIST)',
             color='white', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../static/tsne_perplexity_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved tsne_perplexity_comparison.png')

## 8. Effect of Iterations

In [ ]:
iterations_list = [100, 250, 500, 1000]
iter_results    = {}

print('Running iteration comparison...')
for n_iter in iterations_list:
    print(f'  n_iter={n_iter}...')
    t = TSNE(n_components=2, perplexity=30, n_iter=n_iter,
             learning_rate='auto', init='pca', random_state=42)
    emb = t.fit_transform(X_perp)
    iter_results[n_iter] = (emb, round(float(t.kl_divergence_), 4))
    print(f'    KL divergence: {iter_results[n_iter][1]:.4f}')

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, n_iter in zip(axes, iterations_list):
    emb, kl = iter_results[n_iter]
    for digit in range(10):
        mask = y_perp == digit
        ax.scatter(emb[mask, 0], emb[mask, 1],
                   c=DIGIT_COLORS[digit], s=10, alpha=0.7, edgecolors='none')
    ax.set_title(f'n_iter={n_iter}  KL={kl:.3f}', color=ACCENT, fontsize=11)
    ax.set_xlabel('Dim 1'); ax.set_ylabel('Dim 2')
    ax.grid(True, linestyle='--', alpha=0.15)

plt.suptitle('t-SNE: Effect of Number of Iterations',
             color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Downstream Classification: KNN on t-SNE Embedding

In [ ]:
# KNN on t-SNE 2D
X_tr, X_te, y_tr, y_te = train_test_split(X_tsne, y_sample, test_size=0.2, random_state=42)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_tr, y_tr)
y_pred   = knn.predict(X_te)
knn_acc  = accuracy_score(y_te, y_pred)

# KNN on PCA 50-dim
X_tr2, X_te2, y_tr2, y_te2 = train_test_split(X_pca50, y_sample, test_size=0.2, random_state=42)
knn2 = KNeighborsClassifier(n_neighbors=5)
knn2.fit(X_tr2, y_tr2)
knn_acc_pca = accuracy_score(y_te2, knn2.predict(X_te2))

# KNN on raw scaled
X_tr3, X_te3, y_tr3, y_te3 = train_test_split(X_scaled, y_sample, test_size=0.2, random_state=42)
knn3 = KNeighborsClassifier(n_neighbors=5)
knn3.fit(X_tr3, y_tr3)
knn_acc_raw = accuracy_score(y_te3, knn3.predict(X_te3))

print('KNN Classification Accuracy Comparison:')
print(f'  Raw 784 dims     : {knn_acc_raw*100:.2f}%')
print(f'  PCA 50 dims      : {knn_acc_pca*100:.2f}%')
print(f'  t-SNE 2 dims     : {knn_acc*100:.2f}%')
print()
print('Note: t-SNE is for visualization - low accuracy here is expected.')
print('PCA-50 retains enough structure for classification tasks.')

In [ ]:
print('Classification Report - KNN on t-SNE 2D embedding:')
print(classification_report(y_te, y_pred, target_names=[str(i) for i in range(10)]))

## 10. Evaluation Summary

In [ ]:
print('=' * 55)
print('  t-SNE - Evaluation Summary')
print('=' * 55)
print(f'  Original dimensions      : {X_all.shape[1]}')
print(f'  After PCA pre-reduction  : {PCA_DIMS}  ({pca_var*100:.1f}% var)')
print(f'  After t-SNE              : 2')
print(f'  Perplexity               : 30')
print(f'  Iterations               : 1000')
print(f'  KL Divergence            : {tsne.kl_divergence_:.4f}')
print(f'  KNN acc (t-SNE 2D)       : {knn_acc*100:.2f}%')
print(f'  KNN acc (PCA-50)         : {knn_acc_pca*100:.2f}%')
print(f'  KNN acc (raw 784 dims)   : {knn_acc_raw*100:.2f}%')
print('=' * 55)

## 11. Save Model Artifacts

In [ ]:
os.makedirs('../models', exist_ok=True)

meta = {
    'n_samples_used':      N_SAMPLE,
    'n_features_original': X_all.shape[1],
    'pca_pre_dims':        PCA_DIMS,
    'pca_pre_variance':    round(float(pca_var), 4),
    'tsne_perplexity':     30,
    'tsne_n_iter':         1000,
    'tsne_kl_divergence':  round(float(tsne.kl_divergence_), 4),
    'knn_acc_tsne':        round(knn_acc, 4),
    'knn_acc_pca50':       round(knn_acc_pca, 4),
    'knn_acc_raw':         round(knn_acc_raw, 4),
    'digit_colors':        DIGIT_COLORS,
    'kl_by_perplexity':    kl_divs,
    'class_distribution':  {int(d): int((y_sample==d).sum()) for d in range(10)},
    'pca_2d_variance':     [round(float(v), 4) for v in pca_2d.explained_variance_ratio_],
}

pickle.dump(X_tsne,   open('../models/tsne_embedding.pkl', 'wb'))
pickle.dump(y_sample, open('../models/tsne_labels.pkl',    'wb'))
pickle.dump(pca_pre,  open('../models/pca_pre.pkl',        'wb'))
pickle.dump(pca_2d,   open('../models/pca_2d.pkl',         'wb'))
pickle.dump(scaler,   open('../models/scaler.pkl',         'wb'))
pickle.dump(meta,     open('../models/meta.pkl',           'wb'))

print('Saved:')
print('  models/tsne_embedding.pkl')
print('  models/tsne_labels.pkl')
print('  models/pca_pre.pkl')
print('  models/pca_2d.pkl')
print('  models/scaler.pkl')
print('  models/meta.pkl')

## How to Run the Streamlit App

```bash
pip install -r requirements.txt
python train_model.py
streamlit run app.py
```

---

## Key Takeaways

| Concept | Description |
|---|---|
| t-SNE | Minimizes KL divergence between high-dim and low-dim probability distributions |
| Perplexity | Controls effective neighbourhood size; typical range 5-50 |
| KL Divergence | Objective being minimized; lower = better 2D representation |
| PCA pre-reduction | Reduce to 50 dims before t-SNE to remove noise and speed it up |
| Stochastic | Different random seeds give different layouts; axes have no meaning |
| Visualization only | t-SNE embeddings should not be used for downstream ML tasks |
| PCA vs t-SNE | PCA is linear and fast; t-SNE is nonlinear and shows local structure |
| Limitation | Does not generalize to new points; very slow on large datasets |